# Aula 03 — Parte D: Limpar e Indexar o Corpus do Grupo
**Projeto Integrador III — Motor de Busca (Baixada Santista)**

**Alunos:** Ivan Guimarães dos S. da Silva, Jorge Sugano Suzart e Gustavo Geminiano

*Kernel: R | Google Colaboratory (IRkernel) / GitHub do Grupo*

In [1]:
# 1. Setup oficial do motor03 e carregamento do corpus real do projeto
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor03.R")

REPO <- "https://raw.githubusercontent.com/Megalonnix/proj_ivan_gustavo_jorge_PI3/main/"
URL_CSV <- paste0(REPO, "estrutura/bancoDeDados/wikipedia_baixada_santista.csv")
arquivo_local <- "wikipedia_baixada_santista.csv"

# Download local seguro do CSV no ambiente Colab / R
if (!file.exists(arquivo_local)) {
  download.file(URL_CSV, destfile = arquivo_local, mode = "wb", quiet = TRUE)
}

corpus_raw <- tryCatch({
  read.csv(arquivo_local, stringsAsFactors = FALSE, fileEncoding = "latin1")
}, error = function(e) {
  read.csv(arquivo_local, stringsAsFactors = FALSE, encoding = "latin1")
})

# Configuracao inicial (cfg herdado das Aulas 01-02)
cfg <- list(minimo = 200, grupo = "ivan_gustavo_jorge")
validos <- !is.na(corpus_raw$texto) & (nchar(corpus_raw$texto) > cfg$minimo)
docs   <- corpus_raw$texto[validos]
origem <- ifelse(grepl("guaruja", corpus_raw$titulo[validos], ignore.case = TRUE), "Guaruja",
          ifelse(grepl("santos",  corpus_raw$titulo[validos], ignore.case = TRUE), "Santos", "Bertioga"))

names(docs)   <- paste0("d", 1:length(docs))
names(origem) <- names(docs)

ix <- montar(docs, cfg)

cat("Corpus carregado com sucesso!\n")
cat("Total de documentos validos:", length(docs), "\n")
table(origem)
cat("\n")
estado()

Corpus carregado com sucesso!
Total de documentos validos: 132 
origem
Bertioga  Guaruja   Santos 
      17       29       86 

[1] "MOTOR_VERSAO: motor03 v1 2026-10-01"
| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `cfg` | list | 2 | elementos: minimo, grupo |
| `docs` | character, nomeado | 132 | d1 Guarujá é um município..., d2 Atualmente a cidade de Guarujá..., ... |
| `origem` | character, nomeado | 132 | d1 Guaruja, d2 Guaruja, ..., d76 Santos, ..., d117 Bertioga |
| `ix` | list | 8 | elementos: tokens, vocab, tf, N, df, idf_tfidf, w, wn |


## Módulo 10 — Limpar o corpus do grupo e decisão dos acentos
Avaliação empírica entre as opções `"manter"` e `"translit"` sobre os termos enciclopédicos da Baixada Santista.

In [2]:
# Módulo 10: Funcao limpar com parametro acentos e inspecao comparativa
limpar <- function(x, acentos) {
  if (acentos == "translit") {
    x <- iconv(x, from = "UTF-8", to = "ASCII//TRANSLIT")
  }
  x <- tolower(x)
  x <- gsub("[^a-z0-9 ]", " ", x)
  x <- gsub("\\s+", " ", x)
  trimws(x)
}

tok <- function(x, acentos) {
  t <- unlist(strsplit(limpar(x, acentos), " "))
  t[nzchar(t)]
}

# Comparacao em frase de teste
s <- "O Porto de Santos, inaugurado em 1892, é o maior da América Latina."
cat("Exemplo canonico de comparacao de acentos:\n")
cat("Com manter  : ", limpar(s, "manter"), "\n")
cat("Com translit: ", limpar(s, "translit"), "\n\n")

# Comparacao em d1
cat("Comparacao no documento d1 do corpus:\n")
cat("Total tokens com 'manter'  :", length(tok(docs[[1]], "manter")), "\n")
cat("Total tokens com 'translit':", length(tok(docs[[1]], "translit")), "\n")
cat("Motivo da diferenca: 'manter' parte palavras acentuadas (ex.: 'município' -> 'munic' 'pio').\n\n")

# Verificacao de integridade (deve dar 0 falhas)
erros_translit <- sum(is.na(iconv(docs, from = "UTF-8", to = "ASCII//TRANSLIT")))
cat("Documentos com falha na transliteracao UTF-8:", erros_translit, "(todos validos!)\n")

# Decisao oficial do grupo: 'translit'
cfg$limpar  <- TRUE
cfg$acentos <- "translit"

Exemplo canonico de comparacao de acentos:
Com manter  :  o porto de santos inaugurado em 1892 maior da am rica latina 
Com translit:  o porto de santos inaugurado em 1892 e o maior da america latina 

Comparacao no documento d1 do corpus:
Total tokens com 'manter'  : 65 
Total tokens com 'translit': 61 
Motivo da diferenca: 'manter' parte palavras acentuadas (ex.: 'município' -> 'munic' 'pio').

Documentos com falha na transliteracao UTF-8: 0 (todos validos!)


## Módulo 11 — As stopwords do projeto
Inspeção das frequências empíricas no corpus, definição da lista de stopwords e medição do encolhimento do vocabulário.

In [3]:
# Módulo 11: Frequencias empíricas e definicao de stopwords
todos <- unlist(lapply(docs, function(x) tok(x, cfg$acentos)))
cat("Top 20 termos mais frequentes no corpus limpo:\n")
print(sort(table(todos), decreasing = TRUE)[1:20])

# Lista oficial de stopwords do projeto (27 termos funcionais)
cfg$stopwords <- c(
  "de", "a", "o", "em", "e", "do", "da", "que", "dos", "no",
  "um", "uma", "com", "na", "por", "para", "as", "os", "ao",
  "se", "das", "pelo", "pela", "mais", "foi", "nos", "nas"
)

sem_stop <- function(x, stopwords, acentos) {
  t <- tok(x, acentos)
  t[!t %in% stopwords]
}

tokens_g <- lapply(docs, function(x) sem_stop(x, cfg$stopwords, cfg$acentos))
names(tokens_g) <- names(docs)

v_bruto <- length(ix$vocab)
v_limpo <- length(unique(unlist(tokens_g)))

cat("\nPalavras frequentes mantidas intencionalmente por conterem valor tematico:\n")
cat("santos, sao, cidade, porto, praias, turismo\n\n")

cat("Encolhimento do vocabulario:\n")
cat("Bruto:", v_bruto, "| Limpo (pos translit + stopwords):", v_limpo, "\n")
cat("Reducao total:", v_bruto - v_limpo, "termos eliminados (ruido ortografico, pontuacoes e stopwords).\n")

Top 20 termos mais frequentes no corpus limpo:
    de      a      e      o     do     da     em    que santos    com cidade     na     no    sao    por     um     se   para     os    foi 
   910    562    503    368    302    251    227    164    164    130    129    114    110    107     97     93     89     86     81     78 

Palavras frequentes mantidas intencionalmente por conterem valor tematico:
santos, sao, cidade, porto, praias, turismo

Encolhimento do vocabulario:
Bruto: 4250 | Limpo (pos translit + stopwords): 3483 
Reducao total: 767 termos eliminados (ruido ortografico, pontuacoes e stopwords).


## Módulo 12 — Índice invertido e busca AND no corpus do grupo
Construção da lista de postagens (`postings`) e validação de consultas booleanas conjuntivas sobre o acervo real.

In [4]:
# Módulo 12: Construcao do indice invertido
postings <- list()
for (d in names(tokens_g)) {
  for (termo in unique(tokens_g[[d]])) {
    postings[[termo]] <- c(postings[[termo]], d)
  }
}

ix$postings <- postings

cat("Total de termos indexados no postings:", length(postings), "\n")
cat("Top 5 termos que aparecem em mais documentos (df):\n")
print(sort(lengths(postings), decreasing = TRUE)[1:5])

cat("\nTamanho da lista de postagens de 'porto':", length(postings[["porto"]]), "documentos\n")
cat("Tamanho da lista de postagens de 'santos':", length(postings[["santos"]]), "documentos\n")

Total de termos indexados no postings: 3483 
Top 5 termos que aparecem em mais documentos (df):
santos    sao cidade   como tambem 
    83     71     63     52     38 

Tamanho da lista de postagens de 'porto': 23 documentos
Tamanho da lista de postagens de 'santos': 83 documentos


## Homologação das Consultas de Trabalho e Casos Disjuntos
Execução de `busca_AND()` comparando as postagens reais de cada termo.

In [5]:
# Implementacao oficial de busca_AND
busca_AND <- function(consulta, ix, cfg) {
  termos <- sem_stop(consulta, cfg$stopwords, cfg$acentos)
  if (!all(termos %in% names(ix$postings))) return(character(0))
  Reduce(intersect, ix$postings[termos])
}

# Consulta 1: 'porto santos'
r1 <- busca_AND("porto santos", ix, cfg)
cat("=== Consulta 1: 'porto santos' ===\n")
cat("Documentos encontrados (", length(r1), "): ", paste(r1, collapse = " "), "\n", sep = "")
cat("Distribuicao por cidade de origem:\n")
cat("Santos:", sum(origem[r1] == "Santos"), "| Guaruja:", sum(origem[r1] == "Guaruja"), "| Bertioga:", sum(origem[r1] == "Bertioga"), "\n")
cat("Exemplo Top 1 (d76 - Santos): ", substr(docs[["d76"]], 1, 110), "...\n\n", sep = "")

# Consulta 2: 'praias turismo'
r2 <- busca_AND("praias turismo", ix, cfg)
cat("=== Consulta 2: 'praias turismo' ===\n")
cat("Documentos encontrados (", length(r2), "): ", paste(r2, collapse = " "), "\n", sep = "")
cat("Distribuicao por cidade de origem:\n")
cat("Santos:", sum(origem[r2] == "Santos"), "| Guaruja:", sum(origem[r2] == "Guaruja"), "| Bertioga:", sum(origem[r2] == "Bertioga"), "\n\n")

# Consulta 3: Termos disjuntos sem sobreposicao ('cafe tupinambas')
r3 <- busca_AND("cafe tupinambas", ix, cfg)
cat("=== Consulta 3: Casos Disjuntos (Intersecao Vazia) ===\n")
cat("Busca conjuntiva 'cafe tupinambas':\n")
print(r3)
cat("Explicacao pelas postagens:\n")
cat("Postings de 'cafe'       : ", paste(ix$postings[["cafe"]], collapse = " "), " (origem Santos)\n")
cat("Postings de 'tupinambas' : ", paste(ix$postings[["tupinambas"]], collapse = " "), " (origem Bertioga)\n")
cat("Intersecao das listas: VAZIA (0 documentos em comum).\n")

=== Consulta 1: 'porto santos' ===
Documentos encontrados (21): d9 d25 d27 d28 d30 d36 d39 d41 d45 d49 d50 d51 d75 d76 d77 d78 d85 d95 d99 d124 d130 
Distribuicao por cidade de origem:
Santos: 15 | Guaruja: 4 | Bertioga: 2 
Exemplo Top 1 (d76 - Santos): Santos possui o maior porto da América Latina, o Porto de Santos. Suas obras começaram em 1888...

=== Consulta 2: 'praias turismo' ===
Documentos encontrados (6): d18 d19 d51 d53 d80 d118 
Distribuicao por cidade de origem:
Santos: 3 | Guaruja: 2 | Bertioga: 1 

=== Consulta 3: Casos Disjuntos (Intersecao Vazia) ===
Busca conjuntiva 'cafe tupinambas':
character(0)
Explicacao pelas postagens:
Postings de 'cafe'       : d31 d49 d76 d77 d82 d96 d99 d101 (origem Santos)
Postings de 'tupinambas' : d121 d122 (origem Bertioga)
Intersecao das listas: VAZIA (0 documentos em comum).


## Fechamento da Sessão e Snapshot de Estado
Anexação do estado do R aos consolidados do grupo e ficha de projeto.

In [6]:
# Fechamento e persistencia
if (file.exists("00_FICHA_PROJETO.md")) {
  anexar_estado("00_FICHA_PROJETO.md")
  cat("Estado anexado a 00_FICHA_PROJETO.md\n")
}
if (file.exists("aula03_parteD_consolidado.md")) {
  anexar_estado("aula03_parteD_consolidado.md")
  cat("Estado anexado a aula03_parteD_consolidado.md\n")
}
cat("Sessao da Aula 03 Parte D finalizada com sucesso!\n")

Estado anexado a 00_FICHA_PROJETO.md
Estado anexado a aula03_parteD_consolidado.md
Sessao da Aula 03 Parte D finalizada com sucesso!
